# Topic Modeling

LDA: documents are mixtures of topics, topics are distributions over words.

BERTopic: documents cluster in embedding space, clusters are topics.

## Problem Definition

Figure out what the collection (eg. 5000 news) is about without reading it. You do not have labeled categories, you do not even know how many categories exist.

Topic modeling answers that without supervision. Give it a corpus, get back a small set of coherent topics and, for each document, a distribution over those topics.

## Basic Concept

### LDA

Each topic is a distribution over words. Each document is a mixture of topics. To generate a word in a document, sample a topic from the document's mixture, then sample a word from that topic's distribution per document and the word distribution per topic.

#### Key Output

* Doc_2_Topic Matrix: each row sums to 1, document's topic mixture.
* Topic_2_Word Matrix: each row sums to 1,
topic's word distribution.

### BERTopic pipeline

1. Encode each document with a sentence transformer.
2. Reduce dimensionality with UMAP to ~5 dimensions.
3. Cluster with HDBSCAN. Density-based, produces variable-size clusters and an "outlier" label.
4. For each cluster, compute class-based TF-IDF over the cluster's documents to extract top words.

#### Key Output

One topic per document, plus a -1 outlier label



# Build your Own

## Sample Corpus

In [1]:
import random

random.seed(42)

WORD_BANKS = {
    "technology": {
        "nouns": ["smartphone", "laptop", "tablet", "server", "chip", "app", "browser", "database", "API", "router"],
        "verbs": ["launched", "released", "updated", "patched", "deployed", "optimized", "integrated", "migrated"],
        "adjs": ["faster", "secure", "wireless", "portable", "cloud-based", "modular", "scalable", "open-source"],
        "extras": ["battery life", "user privacy", "machine learning", "edge computing", "data encryption"],
    },
    "sports": {
        "nouns": ["team", "coach", "stadium", "league", "player", "referee", "season", "tournament", "final", "roster"],
        "verbs": ["won", "scored", "defeated", "signed", "traded", "injured", "qualified", "dominated"],
        "adjs": ["dramatic", "overtime", "record-breaking", "comeback", "rival", "championship", "injury-plagued"],
        "extras": ["penalty kick", "three-pointer", "home run", "match point", "playoff berth"],
    },
    "health": {
        "nouns": ["doctor", "patient", "clinic", "vaccine", "therapy", "hospital", "nurse", "symptom", "diet", "sleep"],
        "verbs": ["diagnosed", "treated", "recovered", "prescribed", "monitored", "improved", "prevented", "studied"],
        "adjs": ["chronic", "preventive", "mental", "cardiac", "daily", "balanced", "clinical", "holistic"],
        "extras": ["heart rate", "blood pressure", "immune system", "weight loss", "stress relief"],
    },
    "finance": {
        "nouns": ["stock", "bond", "market", "investor", "bank", "fund", "portfolio", "dividend", "loan", "crypto"],
        "verbs": ["rose", "fell", "traded", "invested", "borrowed", "merged", "listed", "hedged"],
        "adjs": ["volatile", "bullish", "bearish", "quarterly", "annual", "risky", "stable", "global"],
        "extras": ["interest rate", "earnings report", "market cap", "exchange rate", "credit score"],
    },
    "travel": {
        "nouns": ["flight", "hotel", "passport", "airport", "beach", "museum", "tour", "resort", "cruise", "visa"],
        "verbs": ["booked", "visited", "explored", "delayed", "landed", "canceled", "recommended", "discovered"],
        "adjs": ["scenic", "budget", "luxury", "remote", "historic", "coastal", "mountain", "popular"],
        "extras": ["local cuisine", "travel insurance", "boarding gate", "city center", "peak season"],
    },
    "food": {
        "nouns": ["restaurant", "chef", "recipe", "ingredient", "dessert", "menu", "kitchen", "spice", "sauce", "bakery"],
        "verbs": ["cooked", "baked", "grilled", "served", "tasted", "fermented", "seasoned", "roasted"],
        "adjs": ["spicy", "organic", "vegan", "fresh", "homemade", "crispy", "creamy", "smoky"],
        "extras": ["olive oil", "street food", "farm market", "wine pairing", "comfort food"],
    },
    "science": {
        "nouns": ["telescope", "experiment", "lab", "theory", "particle", "galaxy", "sample", "sensor", "orbit", "gene"],
        "verbs": ["observed", "measured", "discovered", "confirmed", "simulated", "published", "tested", "analyzed"],
        "adjs": ["quantum", "molecular", "stellar", "peer-reviewed", "novel", "reproducible", "theoretical"],
        "extras": ["dark matter", "climate data", "space mission", "control group", "research grant"],
    },
    "entertainment": {
        "nouns": ["movie", "album", "series", "actor", "director", "concert", "game", "festival", "trailer", "studio"],
        "verbs": ["streamed", "premiered", "nominated", "reviewed", "performed", "released", "watched", "ranked"],
        "adjs": ["indie", "blockbuster", "animated", "live-action", "viral", "award-winning", "binge-worthy"],
        "extras": ["box office", "opening night", "soundtrack", "fan theory", "streaming platform"],
    },
    "politics": {
        "nouns": ["election", "senator", "policy", "debate", "vote", "cabinet", "bill", "campaign", "summit", "treaty"],
        "verbs": ["passed", "debated", "elected", "proposed", "vetoed", "negotiated", "rallied", "approved"],
        "adjs": ["bipartisan", "controversial", "federal", "local", "diplomatic", "urgent", "historic"],
        "extras": ["public opinion", "foreign policy", "tax reform", "voter turnout", "press briefing"],
    },
    "environment": {
        "nouns": ["forest", "ocean", "species", "emission", "recycling", "wetland", "drought", "glacier", "pollution", "habitat"],
        "verbs": ["protected", "restored", "reduced", "monitored", "threatened", "conserved", "planted", "cleaned"],
        "adjs": ["renewable", "endangered", "sustainable", "toxic", "carbon-neutral", "biodiverse", "fragile"],
        "extras": ["solar panel", "carbon footprint", "plastic waste", "clean energy", "wildlife corridor"],
    },
}

PATTERNS = [
    "A {adj} {noun} was {verb} after teams focused on {extra}.",
    "Experts say the {noun} could change how people handle {extra}.",
    "Local news covered a {adj} {noun} linked to rising {extra}.",
    "Analysts {verb} the latest {noun} amid debate over {extra}.",
    "Communities welcomed a {adj} plan to improve {extra} this year.",
    "Reports show the {noun} has {verb} interest in {extra} recently.",
    "Officials {verb} a {adj} {noun} designed to support {extra}.",
    "Fans praised a {adj} update to the {noun} featuring {extra}.",
    "Researchers {verb} data from a {noun} study about {extra}.",
    "The {noun} market saw {adj} movement after {extra} headlines.",
]

def make_document(topic: str) -> str:
    bank = WORD_BANKS[topic]
    sentence = random.choice(PATTERNS).format(
        adj=random.choice(bank["adjs"]),
        noun=random.choice(bank["nouns"]),
        verb=random.choice(bank["verbs"]),
        extra=random.choice(bank["extras"]),
    )
    return " ".join(sentence.split()[:20])

topics = list(WORD_BANKS.keys())
documents = [make_document(random.choice(topics)) for _ in range(1000)]

assert len(documents) == 1000
assert all(len(doc.split()) <= 20 for doc in documents)
print(f"Generated {len(documents)} documents (max {max(len(d.split()) for d in documents)} words)")
print("Examples:")
for doc in documents[:3]:
    print("-", doc)

Generated 1000 documents (max 11 words)
Examples:
- A championship player was signed after teams focused on three-pointer.
- Experts say the sleep could change how people handle heart rate.
- Experts say the server could change how people handle data encryption.


## LDA via scikit-learn

In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import numpy as np

def fit_lda(documents, n_topics=5, max_features=1000):
    cv = CountVectorizer(
        max_features=max_features,
        stop_words="english",
        min_df=2,
        max_df=0.9
    )
    X = cv.fit_transform(documents)

    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        max_iter=50,
        learning_method="online"
    )
    doc_topic = lda.fit_transform(X)

    feature_names = cv.get_feature_names_out()

    return lda, cv, doc_topic, feature_names

def print_top_words(lda, feature_names, n_top=10):
    for idx, topic in enumerate(lda.components_):
        top_idx = np.argsort(-topic)[:n_top]
        words = [feature_names[i] for i in top_idx]
        print(f"Topic {idx}: {', '.join(words)}")

lda, cv, doc_topic, feature_names = fit_lda(documents)
print_top_words(lda, feature_names)


Topic 0: market, movement, headlines, saw, update, featuring, praised, fans, season, grant
Topic 1: debate, analysts, latest, amid, data, researchers, study, rate, reports, recently
Topic 2: experts, change, people, say, handle, focused, teams, theory, press, briefing
Topic 3: local, year, communities, improve, welcomed, plan, linked, news, rising, covered
Topic 4: support, officials, designed, study, researchers, data, reports, recently, policy, food


## BERTopic

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    min_topic_size=15,
    verbose=True
)

topics, probs = topic_model.fit_transform(documents)
info = topic_model.get_topic_info()
print(info.head(20))

valid_topics = info[info["Topic"] != -1]["Topic"].tolist()
for topic_id in valid_topics[:5]:
    print(f"Topic {topic_id}: {topic_model.get_topic(topic_id)[:10]}")

################################# Keywords #################################
for topic_id in valid_topics:
    words = [w for w, _ in topic_model.get_topic(topic_id)[:5]]
    print(f"Topic {topic_id} ({info.loc[info.Topic==topic_id, 'Count'].values[0]} docs): {', '.join(words)}")

W0804 23:20:43.606000 69160 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0804 23:20:43.630000 69160 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0804 23:20:43.650000 69160 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-08-04 23:20:44,699 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

2026-08-04 23:20:54,763 - BERTopic - Embedding - Completed ✓
2026-08-04 23:20:54,764 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-04 23:20:58,919 - BERTopic - Dimensionality - Completed ✓
2026-08-04 23:20:58,920 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-04 23:20:58,934 - BERTopic - Cluster - Completed ✓
2026-08-04 23:20:58,939 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-04 23:20:58,957 - BERTopic - Representation - Completed ✓


    Topic  Count                               Name  \
0      -1      8      -1_stellar_year_welcomed_this   
1       0    111     0_carbon_energy_clean_corridor   
2       1    104    1_policy_foreign_briefing_press   
3       2    102        2_food_comfort_wine_pairing   
4       3    100        3_relief_stress_weight_loss   
5       4    100     4_group_control_grant_research   
6       5     94         5_fan_theory_night_opening   
7       6     92  6_battery_life_encryption_machine   
8       7     90       7_rate_earnings_report_score   
9       8     82       8_kick_penalty_berth_playoff   
10      9     81          9_boarding_gate_peak_city   
11     10     19    10_three_pointer_plagued_injury   
12     11     17  11_plan_communities_welcomed_this   

                                       Representation  \
0   [stellar, year, welcomed, this, communities, p...   
1   [carbon, energy, clean, corridor, wildlife, so...   
2   [policy, foreign, briefing, press, public, opi...   
3